# MobilityDB × PostGIS Raster

This notebook demonstrates the raster sampling operator family shipped with
MobilityDB when built with `-DRASTER=ON`. It runs against **two backends**:

| Backend | How to select | Raster sampling |
|---|---|---|
| **PostgreSQL + MobilityDB** | `MOBILITY_BACKEND=pg` *(default)* | C operators (`raster_value`, …) |
| **DuckDB + MobilityDuck** | `MOBILITY_BACKEND=duckdb` | Python/numpy equivalents |

Trip data is always read from PostgreSQL (BerlinMOD `berlinmod_h3bench` DB).

| Operator | Signature | Returns |
|---|---|---|
| `raster_value` | `(raster, tgeompoint, band int DEFAULT 1)` | `tfloat` |
| `atRasterValue` | `(tgeompoint, raster, floatspan, band int DEFAULT 1)` | `tgeompoint` |
| `minusRasterValue` | `(tgeompoint, raster, floatspan, band int DEFAULT 1)` | `tgeompoint` |
| `eRasterValue` | `(raster, tgeompoint, floatspan, band int DEFAULT 1)` | `boolean` |
| `aRasterValue` | `(raster, tgeompoint, floatspan, band int DEFAULT 1)` | `boolean` |
| `trajectory_quadbins` | `(tgeompoint, zoom int)` | `bigint[]` |
| `raster_tile_value_quadbin` | `(bytea, w, h, cell, dtype, nodata, le, tgeompoint)` | `tfloat` |


In [ ]:
# Install the notebook's Python dependencies (safe to re-run; needed on a fresh kernel / Colab).
%pip install -q "psycopg[binary]" pymeos numpy pandas matplotlib pillow duckdb quadbin shapely


## Setup

Set `MOBILITY_BACKEND` **before** starting the kernel (or in the cell below before
running the rest of the notebook):

```bash
# shell — choose one:
export MOBILITY_BACKEND=pg       # PostgreSQL + MobilityDB with -DRASTER=ON (default)
export MOBILITY_BACKEND=duckdb   # DuckDB + MobilityDuck + Python raster helpers
```

**Prerequisites per backend:**

| | `"pg"` | `"duckdb"` |
|---|---|---|
| Database | PostgreSQL with MobilityDB (`-DRASTER=ON`) | PostgreSQL with MobilityDB (trip data only) |
| Extension | `postgis_raster` (auto-installed via `CASCADE`) | `mobilityduck` (DuckDB community) |
| Python extras | — | `quadbin`, `shapely` (installed by cell above) |
| Section 8 | `trajectory_quadbins` + `raster_tile_value_quadbin` (C) | `quadbin.geometry_to_cells` + `struct.unpack_from` |


In [ ]:
import os

BACKEND    = os.environ.get('MOBILITY_BACKEND', 'pg')   # 'pg' or 'duckdb'
PGHOST     = os.environ.get('PGHOST',     '/tmp')
PGPORT     = os.environ.get('PGPORT',     '5432')
PGDATABASE = os.environ.get('PGDATABASE', 'berlinmod_h3bench')
DSN = f'host={PGHOST} port={PGPORT} dbname={PGDATABASE}'

import psycopg
conn = psycopg.connect(DSN, autocommit=True)
cur  = conn.cursor()
def q1(sql, args=None):
    with conn.cursor() as c:
        c.execute(sql, args); r = c.fetchone(); return r[0] if r else None

print(f'Backend : {BACKEND}')
print(f'Database: {conn.info.dbname}  (MobilityDB {q1("SELECT mobilitydb_version()")})')

if BACKEND == 'pg':
    assert q1("SELECT 1 FROM pg_available_extensions WHERE name = 'postgis_raster'"), \
        'postgis_raster not available — rebuild MobilityDB with -DRASTER=ON'
    print('PostGIS raster: present')
else:
    import duckdb as _ddb
    duck = _ddb.connect()
    duck.execute('INSTALL spatial; LOAD spatial')
    duck.execute('INSTALL mobilityduck; LOAD mobilityduck')
    print('DuckDB / MobilityDuck: ready')


In [ ]:
import math
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.collections as mcoll
import pandas as pd
from PIL import Image

from pymeos import *
pymeos_initialize()

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# OSM XYZ-tile basemap helper.
# BerlinMOD trips are already in SRID 3857 (Web Mercator = OSM tile CRS).
_WORLD = 20037508.342789244
_TILE_CACHE = '/tmp/temporal_raster_tiles'
os.makedirs(_TILE_CACHE, exist_ok=True)

def _fetch_tile(z, x, y):
    p = f'{_TILE_CACHE}/{z}_{x}_{y}.png'
    if not os.path.exists(p):
        req = urllib.request.Request(
            f'https://tile.openstreetmap.org/{z}/{x}/{y}.png',
            headers={'User-Agent': 'mobilitydb-temporal-raster/1.0'})
        open(p, 'wb').write(urllib.request.urlopen(req, timeout=25).read())
    return Image.open(p).convert('RGB')

def _m2t(x, y, z):
    n = 2 ** z
    return (x + _WORLD) / (2 * _WORLD) * n, (_WORLD - y) / (2 * _WORLD) * n

def _t2m(tx, ty, z):
    n = 2 ** z
    return tx / n * 2 * _WORLD - _WORLD, _WORLD - ty / n * 2 * _WORLD

def add_basemap(ax, xmin, ymin, xmax, ymax):
    span = max(xmax - xmin, ymax - ymin)
    z = 12
    for zz in range(3, 17):
        if span / (2 * _WORLD / 2 ** zz) >= 4:
            z = zz
            break
    tx0, ty1 = _m2t(xmin, ymin, z)
    tx1, ty0 = _m2t(xmax, ymax, z)
    x0, x1 = math.floor(tx0), math.ceil(tx1)
    y0, y1 = math.floor(ty0), math.ceil(ty1)
    mosaic = Image.new('RGB', ((x1 - x0) * 256, (y1 - y0) * 256), (238, 238, 238))
    for xi in range(x0, x1):
        for yi in range(y0, y1):
            try:
                mosaic.paste(_fetch_tile(z, xi, yi), ((xi - x0) * 256, (yi - y0) * 256))
            except Exception:
                pass
    mx0, my1 = _t2m(x0, y0, z)
    mx1, my0 = _t2m(x1, y1, z)
    ax.imshow(mosaic, extent=[mx0, mx1, my0, my1], zorder=0)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

## 1. Elevation Raster

Create a synthetic elevation raster covering the BerlinMOD area. MobilityDB
built with `-DRASTER=ON` declares `postgis_raster` as a dependency, so
`CREATE EXTENSION mobilitydb CASCADE` installs it automatically. The
`raster_value` family are C functions in the extension — no SQL wrappers needed.

In [ ]:
# ── Raster grid parameters (shared by both backends) ─────────────────────────
RAST_W, RAST_H     = 40, 35
RAST_ULX, RAST_ULY = 464000.0, 6610000.0   # upper-left corner in SRID 3857
RAST_SX,  RAST_SY  = 1000.0,  -1000.0       # metres/pixel (neg = top-down)

# Three-Gaussian synthetic elevation (same formula as the PostGIS raster below)
_xi = np.arange(1, RAST_W + 1, dtype=float)
_yi = np.arange(1, RAST_H + 1, dtype=float)
_XI, _YI = np.meshgrid(_xi, _yi)
ELEV_GRID = np.clip(
    30.0
    + 220.0 * np.exp(-((_XI - 20)**2 + (_YI -  8)**2) / 80.0)
    + 160.0 * np.exp(-((_XI - 35)**2 + (_YI - 25)**2) / 60.0)
    + 110.0 * np.exp(-((_XI -  8)**2 + (_YI - 28)**2) / 45.0),
    10.0, 350.0
).astype(np.float32)

# matplotlib raster extent [left, right, bottom, top] in SRID 3857 metres
img_extent = [RAST_ULX,
              RAST_ULX + RAST_W * RAST_SX,
              RAST_ULY + RAST_H * RAST_SY,
              RAST_ULY]

def _sample_elev(x_3857, y_3857):
    """Sample ELEV_GRID at an SRID-3857 coordinate; None if outside extent."""
    col = int((x_3857 - RAST_ULX) / RAST_SX)
    row = int((y_3857 - RAST_ULY) / RAST_SY)
    if 0 <= row < RAST_H and 0 <= col < RAST_W:
        return float(ELEV_GRID[row, col])
    return None

def _raster_value_py(trip_text):
    """Python implementation of raster_value(rast, trip) → (times, values)."""
    trip = TGeomPointSeq(trip_text)
    times, values = [], []
    for inst in trip.instants():
        pt   = inst.value()
        elev = _sample_elev(pt.x, pt.y)
        if elev is not None:
            times.append(inst.timestamp())
            values.append(elev)
    return times, values

if BACKEND == 'pg':
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis_raster")
    cur.execute("DROP TABLE IF EXISTS demo_elevation")
    cur.execute("""
CREATE TABLE demo_elevation (rid serial PRIMARY KEY, rast raster);
INSERT INTO demo_elevation (rast)
SELECT ST_SetValues(
    ST_AddBand(
        ST_MakeEmptyRaster(40, 35, 464000.0, 6610000.0, 1000.0, -1000.0, 0.0, 0.0, 3857),
        '32BF'::text, 0.0::float8, NULL::float8),
    1, 1, 1,
    ARRAY(
        SELECT ARRAY(
            SELECT GREATEST(10.0, LEAST(350.0,
                30.0
                + 220.0 * exp(-((xi-20)::float^2 + (yi- 8)::float^2) / 80.0)
                + 160.0 * exp(-((xi-35)::float^2 + (yi-25)::float^2) / 60.0)
                + 110.0 * exp(-((xi- 8)::float^2 + (yi-28)::float^2) / 45.0)
            ))::float4
            FROM generate_series(1, 40) xi
        )
        FROM generate_series(1, 35) yi
    )
);
""")
    print('Raster created in PG: 40x35 pixels, 1 km resolution, SRID 3857')
else:
    print('Elevation grid ready (numpy 40x35, 1 km resolution, SRID 3857)')
    print(f'Elevation range: {ELEV_GRID.min():.1f} – {ELEV_GRID.max():.1f} m')


## 2. Elevation Profile: Single Trip

`raster_value(rast, trip)` evaluates the raster band at every GPS fix of the
trajectory and returns a `tfloat`. Instants outside the raster extent are
silently dropped. The result composes directly with the full `tfloat` operator
family (`minValue`, `maxValue`, `avgValue`, `atValues`, `getTime`, …).

In [ ]:
if BACKEND == 'pg':
    cur.execute("""
SELECT raster_value(rast, trip)::text AS elevation_profile
FROM demo_elevation, trips
WHERE rid = 1 AND tripid = 1
""")
    elev_str = cur.fetchone()[0]
    print(f"Result type: tfloat ({elev_str[:80]}...)")
    elev_tf = TFloatSeq(elev_str)
    insts  = list(elev_tf.instants())
    times  = [i.timestamp() for i in insts]
    values = [i.value()     for i in insts]
else:
    cur.execute("SELECT trip::text FROM trips WHERE tripid = 1")
    times, values = _raster_value_py(cur.fetchone()[0])
    print(f"Result: {len(times)} sampled instants (_raster_value_py / numpy)")
print(f"Instants: {len(times)}  |  Min: {min(values):.1f} m  |  Max: {max(values):.1f} m")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_between(times, values, min(values), alpha=0.25, color='steelblue')
ax.plot(times, values, color='steelblue', linewidth=1.2)
ax.set_xlabel('Time')
ax.set_ylabel('Elevation (m)')
ax.set_title('Elevation profile along trip 1  — raster_value(demo_elevation, trip) → tfloat')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Raster + Trajectories

Overlay the elevation raster with a sample of vehicle paths. The raster is
retrieved pixel-by-pixel from PostgreSQL and plotted as a heatmap; each
trajectory is reconstructed from its BerlinMOD geometry linestring.

In [ ]:
if BACKEND == 'pg':
    # Fetch raster pixel values from PG
    cur.execute("""
SELECT col, row, ST_Value(rast, 1, col, row) AS val,
       ST_UpperLeftX(rast) AS ulx, ST_UpperLeftY(rast) AS uly,
       ST_ScaleX(rast) AS sx, ST_ScaleY(rast) AS sy
FROM demo_elevation,
     generate_series(1, ST_Width(rast))  col,
     generate_series(1, ST_Height(rast)) row
ORDER BY row, col
""")
    pix = cur.fetchall()
    elev_grid = np.array([p[2] for p in pix], dtype=float).reshape(RAST_H, RAST_W)
    # Mean elevation per vehicle via C raster_value
    cur.execute("""
SELECT t.vehicleid,
       avg(avgValue(raster_value(d.rast, t.trip)))::float AS mean_elev
FROM demo_elevation d, trips t
WHERE d.rid = 1 AND t.vehicleid <= 30
GROUP BY t.vehicleid
ORDER BY t.vehicleid
""")
    mean_elev = {row[0]: row[1] for row in cur.fetchall()}
else:
    elev_grid = ELEV_GRID.astype(float)
    # Mean elevation per vehicle via Python helper
    cur.execute("SELECT vehicleid, trip::text FROM trips WHERE vehicleid <= 30")
    _veh_elevs = {}
    for vid, trip_text in cur.fetchall():
        _, vals = _raster_value_py(trip_text)
        if vals:
            _veh_elevs.setdefault(vid, []).extend(vals)
    mean_elev = {vid: float(np.mean(v)) for vid, v in _veh_elevs.items()}
# Trajectory polylines (same for both backends)
cur.execute("""
SELECT vehicleid, ST_X(dp.geom), ST_Y(dp.geom)
FROM (SELECT DISTINCT vehicleid FROM trips WHERE vehicleid <= 30) v
JOIN trips t USING (vehicleid)
JOIN LATERAL ST_DumpPoints(trajectory) dp ON true
WHERE tripid IN (SELECT tripid FROM trips WHERE vehicleid=t.vehicleid LIMIT 1)
ORDER BY vehicleid, dp.path
""")
vehicles = {}
for vid, x, y in cur.fetchall():
    vehicles.setdefault(vid, []).append((x, y))
print(f'Mean elevation range: {min(mean_elev.values()):.1f} – {max(mean_elev.values()):.1f} m')


In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(elev_grid, cmap='terrain', origin='upper',
               extent=img_extent, aspect='equal', alpha=0.75)
plt.colorbar(im, ax=ax, label='Elevation (m)', fraction=0.03)

elev_vals = list(mean_elev.values())
norm = mcolors.Normalize(vmin=min(elev_vals), vmax=max(elev_vals))
cmap_traj = plt.colormaps['RdYlGn_r']

for vid, pts in vehicles.items():
    xs, ys = zip(*pts)
    col = cmap_traj(norm(mean_elev.get(vid, 150)))
    ax.plot(xs, ys, color=col, linewidth=1.0, alpha=0.85)
    ax.plot(xs[0], ys[0], 'o', color=col, markersize=4)

sm = plt.cm.ScalarMappable(cmap=cmap_traj, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Mean sampled elevation (m)', fraction=0.03, pad=0.01)
ax.set_xlabel('Easting (m, SRID 3857)')
ax.set_ylabel('Northing (m, SRID 3857)')
ax.set_title('30 BerlinMOD trips coloured by mean elevation from raster_value()')
plt.tight_layout()
plt.show()

## 4. Fleet Elevation Statistics

Apply `raster_value()` to all 1,620 trips and compute per-vehicle aggregate
statistics on the resulting `tfloat` values: minimum, maximum, and
time-weighted average elevation encountered during the trip.

MobilityDB provides `minValue()`, `maxValue()`, and `avgValue()` on `tfloat`
directly — no manual aggregation needed.

In [ ]:
if BACKEND == 'pg':
    cur.execute("""
WITH elev AS (
    SELECT vehicleid, tripid,
           raster_value(rast, trip) AS trip_elev
    FROM demo_elevation, trips
    WHERE rid = 1
)
SELECT vehicleid,
       count(*)                                     AS trips,
       min(minValue(trip_elev))::numeric(6,1)       AS min_elev_m,
       max(maxValue(trip_elev))::numeric(6,1)       AS max_elev_m,
       avg(avgValue(trip_elev))::numeric(6,1)       AS mean_elev_m
FROM elev
WHERE trip_elev IS NOT NULL
GROUP BY vehicleid
ORDER BY max_elev_m DESC
LIMIT 15
""")
    df = pd.DataFrame(cur.fetchall(),
                      columns=['vehicleid','trips','min_elev','max_elev','mean_elev'])
else:
    cur.execute("SELECT vehicleid, tripid, trip::text FROM trips ORDER BY vehicleid, tripid")
    _stats = {}
    for vid, tid, trip_text in cur.fetchall():
        _, vals = _raster_value_py(trip_text)
        if vals:
            s = _stats.setdefault(vid, {'trips':0,'mins':[],'maxs':[],'means':[]})
            s['trips'] += 1; s['mins'].append(min(vals))
            s['maxs'].append(max(vals)); s['means'].append(float(np.mean(vals)))
    _rows = [(v, s['trips'], min(s['mins']), max(s['maxs']), float(np.mean(s['means'])))
             for v, s in _stats.items()]
    df = pd.DataFrame(_rows, columns=['vehicleid','trips','min_elev','max_elev','mean_elev'])
    df = df.sort_values('max_elev', ascending=False).head(15)
df[['min_elev','max_elev','mean_elev']] = df[['min_elev','max_elev','mean_elev']].astype(float)
print(df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df))
ax.bar(x, df['max_elev'] - df['min_elev'], bottom=df['min_elev'],
       color='steelblue', alpha=0.6, label='elevation range')
ax.plot(x, df['mean_elev'], 'o-', color='firebrick', markersize=5, label='time-weighted avg')
ax.set_xticks(x)
ax.set_xticklabels([f"v{vid}" for vid in df['vehicleid']], rotation=45, ha='right')
ax.set_ylabel('Elevation (m)')
ax.set_title('Per-vehicle elevation range and time-weighted average (top-15 by max elevation)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Elevation Heatmaps: Spatial Insight Across the Fleet

`raster_value()` returns a `tfloat` at the same timestamps as the input
trajectory's instants. To recover (x, y, elevation) tuples we restrict the
`tgeompoint` to the times covered by the `tfloat` result using
`atTime(trip, getTime(trip_elev))`, then unnest both and pair by timestamp.

- **Left** — each GPS fix coloured by sampled elevation (green = low, red = high).
- **Right** — hexbin density of fixes above 150 m, identifying high-ground districts.

In [ ]:
if BACKEND == 'pg':
    cur.execute("""
WITH rv AS (
    SELECT vehicleid, trip,
           raster_value(rast, trip) AS trip_elev
    FROM demo_elevation, trips
    WHERE rid = 1 AND vehicleid <= 20
)
SELECT ST_X(getValue(g))::float AS x,
       ST_Y(getValue(g))::float AS y,
       getValue(e)::float        AS elev
FROM rv,
     LATERAL (
         SELECT g, e
         FROM unnest(instants(atTime(trip, getTime(trip_elev)))) g
         CROSS JOIN unnest(instants(trip_elev)) e
         WHERE getTimestamp(g) = getTimestamp(e)
     ) paired
WHERE trip_elev IS NOT NULL
""")
    pts = cur.fetchall()
else:
    cur.execute("SELECT trip::text FROM trips WHERE vehicleid <= 20")
    pts = []
    for (trip_text,) in cur.fetchall():
        trip = TGeomPointSeq(trip_text)
        for inst in trip.instants():
            pt   = inst.value()
            elev = _sample_elev(pt.x, pt.y)
            if elev is not None:
                pts.append((pt.x, pt.y, elev))
gx = np.array([p[0] for p in pts])
gy = np.array([p[1] for p in pts])
ge = np.array([p[2] for p in pts])
print(f'{len(pts):,} GPS fixes with elevation data')
print(f'Elevation range: {ge.min():.1f} – {ge.max():.1f} m')


In [ ]:
margin = 1500
xmin, xmax = gx.min() - margin, gx.max() + margin
ymin, ymax = gy.min() - margin, gy.max() + margin

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

norm = mcolors.Normalize(vmin=ge.min(), vmax=ge.max())
cmap = plt.colormaps['RdYlGn_r']

# Left: GPS fixes coloured by elevation
ax = axes[0]
add_basemap(ax, xmin, ymin, xmax, ymax)
sc = ax.scatter(gx, gy, c=ge, cmap=cmap, norm=norm, s=1.2, alpha=0.65,
                rasterized=True, zorder=2)
fig.colorbar(sc, ax=ax, label='Sampled elevation (m)', fraction=0.035)
ax.set_title('GPS fixes coloured by raster_value()\n20 vehicles · all trips')
ax.set_axis_off()

# Right: hexbin density of high-elevation fixes
ax = axes[1]
add_basemap(ax, xmin, ymin, xmax, ymax)
mask = ge > 150
if mask.sum() > 0:
    hb = ax.hexbin(gx[mask], gy[mask], gridsize=28, cmap='YlOrRd',
                   mincnt=1, alpha=0.80, zorder=2)
    fig.colorbar(hb, ax=ax, label='Fix count (elev > 150 m)', fraction=0.035)
ax.set_title('Spatial density of fixes above 150 m\n(high-ground heatmap)')
ax.set_axis_off()

fig.suptitle('Fleet-level elevation analytics via raster_value(rast, trip) → tfloat',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 6. GPS Fixes Above an Elevation Threshold

Because `raster_value()` returns a discrete `tfloat` (one value per GPS fix),
threshold analysis is count-based. MobilityDB's `numInstants()`, `atValues()`,
and `maxValue()` operate directly on the `tfloat` result — no additional raster
queries needed.

In [ ]:
if BACKEND == 'pg':
    cur.execute("""
WITH elev AS (
    SELECT vehicleid, tripid,
           raster_value(rast, trip) AS trip_elev
    FROM demo_elevation, trips
    WHERE rid = 1 AND vehicleid = 1
)
SELECT vehicleid, tripid,
       numInstants(trip_elev)                                       AS total_fixes,
       numInstants(atValues(trip_elev, floatspan '[150, 9999]'))    AS fixes_above_150m,
       round(maxValue(atValues(trip_elev,
             floatspan '[150, 9999]'))::numeric, 1)                 AS peak_m
FROM elev
WHERE trip_elev IS NOT NULL
  AND atValues(trip_elev, floatspan '[150, 9999]') IS NOT NULL
ORDER BY tripid
""")
    for vid, tid, total, above, peak in cur.fetchall():
        pct = 100.0 * above / total if total else 0
        print(f"Trip {tid}: {above}/{total} fixes above 150 m ({pct:.0f}%)  ·  peak {peak:.1f} m")
else:
    cur.execute("SELECT vehicleid, tripid, trip::text FROM trips WHERE vehicleid = 1")
    for vid, tid, trip_text in cur.fetchall():
        _, vals = _raster_value_py(trip_text)
        above_vals = [v for v in vals if v >= 150]
        if above_vals:
            pct = 100.0 * len(above_vals) / len(vals)
            print(f"Trip {tid}: {len(above_vals)}/{len(vals)} fixes above 150 m "
                  f"({pct:.0f}%)  ·  peak {max(above_vals):.1f} m")


## 7. Restriction and Predicate Operators

Four SQL-defined operators compose `raster_value` with the MobilityDB temporal
restriction and predicate functions. Each calls `raster_value` exactly once.

| Operator | Semantics |
|---|---|
| `atRasterValue(traj, rast, vspan)` | Sub-trajectory at instants **inside** the value range |
| `minusRasterValue(traj, rast, vspan)` | Sub-trajectory at instants **outside** the value range |
| `eRasterValue(rast, traj, vspan)` | `true` if **ever** inside the value range |
| `aRasterValue(rast, traj, vspan)` | `true` if **always** inside the value range |

In [ ]:
if BACKEND == 'pg':
    cur.execute("""
SELECT tripid,
       numInstants(trip)                                               AS total_fixes,
       numInstants(atRasterValue(trip, rast, floatspan '[100, 9999]'))    AS above_100m,
       numInstants(minusRasterValue(trip, rast, floatspan '[100, 9999]')) AS below_100m
FROM demo_elevation, trips
WHERE rid = 1 AND vehicleid = 1
ORDER BY tripid
LIMIT 10
""")
    df_restr = pd.DataFrame(cur.fetchall(),
                            columns=['tripid','total','above_100m','below_100m'])
else:
    cur.execute("SELECT tripid, trip::text FROM trips WHERE vehicleid = 1 "
                "ORDER BY tripid LIMIT 10")
    _rows = []
    for tid, trip_text in cur.fetchall():
        _, vals = _raster_value_py(trip_text)
        total = len(vals)
        above = sum(1 for v in vals if v >= 100)
        _rows.append((tid, total, above, total - above))
    df_restr = pd.DataFrame(_rows, columns=['tripid','total','above_100m','below_100m'])
print(df_restr.to_string(index=False))


In [ ]:
if BACKEND == 'pg':
    cur.execute("""
SELECT vehicleid, tripid
FROM demo_elevation, trips
WHERE rid = 1
  AND eRasterValue(rast, trip, floatspan '[170, 9999]')
ORDER BY vehicleid, tripid
LIMIT 10
""")
    rows = cur.fetchall()
    print(f'{len(rows)} trips ever above 170 m (first 10 shown):')
    for r in rows:
        print(f'  vehicle {r[0]:3d}, trip {r[1]}')
    cur.execute("""
SELECT count(*)
FROM demo_elevation, trips
WHERE rid = 1
  AND aRasterValue(rast, trip, floatspan '[0, 50]')
""")
    print(f"\nTrips that stayed entirely below 50 m: {cur.fetchone()[0]}")
else:
    cur.execute("SELECT vehicleid, tripid, trip::text FROM trips ORDER BY vehicleid, tripid")
    ever_above, always_below = [], 0
    for vid, tid, trip_text in cur.fetchall():
        _, vals = _raster_value_py(trip_text)
        if vals:
            if max(vals) > 170:
                ever_above.append((vid, tid))
            if max(vals) <= 50:
                always_below += 1
    print(f'{len(ever_above)} trips ever above 170 m (first 10 shown):')
    for r in ever_above[:10]:
        print(f'  vehicle {r[0]:3d}, trip {r[1]}')
    print(f'\nTrips that stayed entirely below 50 m: {always_below}')


## 8. Raquet / QUADBIN

[Raquet](https://github.com/CartoDB/raquet) is a cloud-native raster format
that stores tiles as rows in Apache Parquet, each keyed by a
[QUADBIN](https://docs.carto.com/data-and-analysis/analytics-toolbox-for-bigquery/sql-reference/quadbin)
cell — a 64-bit integer encoding zoom level and Morton-interleaved x/y
coordinates. No external metadata is required: the tile bounding box and
pixel-to-coordinate mapping are fully determined by the cell value.

The typical query pattern is a two-step join:
1. `trajectory_quadbins(traj4326, zoom)` — find which tiles the trajectory overlaps
2. `raster_tile_value_quadbin(...)` — sample each matching tile's pixel bytes

Both functions expect SRID 4326. BerlinMOD trips are in SRID 3857, so we cast
with `ST_Transform`.

```sql
-- Canonical Raquet join pattern
SELECT t.tripid,
       raster_tile_value_quadbin(
           r.band_data, r.width, r.height, r.quadbin, 'FLOAT32', r.nodata, true,
           ST_Transform(t.trip, 4326)::tgeompoint)
FROM trips t
JOIN elevation_raquet r
  ON r.quadbin = ANY(trajectory_quadbins(ST_Transform(t.trip, 4326)::tgeompoint, 8));
```

In [ ]:
# Count of QUADBIN tiles at zoom 10 touched by each trip (vehicles 1–5)
if BACKEND == 'pg':
    cur.execute("""
    SELECT vehicleid, tripid,
           array_length(
               trajectory_quadbins(
                   ST_Transform(trip, 4326)::tgeompoint, 10
               ), 1
           ) AS tile_count
    FROM trips
    WHERE vehicleid <= 5
    ORDER BY vehicleid, tripid
    LIMIT 15
    """)
    for vid, tid, tc in cur.fetchall():
        print(f'  vehicle {vid:2d}  trip {tid:3d}  →  {tc} tiles at zoom 10')
else:
    # DuckDB path: get trajectory linestrings from PG via psycopg, then
    # compute covered QUADBIN cells with the Python quadbin library.
    import json as _json
    import quadbin as _qb
    from shapely import wkt as _swkt

    cur.execute("""
        SELECT vehicleid, tripid,
               ST_AsText(trajectory(ST_Transform(trip, 4326)::tgeompoint)) AS traj_wkt
        FROM trips
        WHERE vehicleid <= 5
        ORDER BY vehicleid, tripid
        LIMIT 15
    """)
    for vid, tid, traj_wkt in cur.fetchall():
        geom  = _swkt.loads(traj_wkt)
        cells = _qb.geometry_to_cells(_json.dumps(geom.__geo_interface__), 10)
        print(f'  vehicle {vid:2d}  trip {tid:3d}  →  {len(cells)} tiles at zoom 10')


In [ ]:
# raster_tile_value_quadbin: sample a synthetic 2×2 UINT8 chip.
# In production, pixel bytes come from a Raquet Parquet table
# (e.g. via pandas + duckdb: df.loc[df['quadbin'].isin(tiles), 'band_data']).
#
# Tile (x=1, y=0, zoom=1): lon 0–180°, lat 0–85.05° (north Mercator half)
# Chip layout (linear interpolation within tile bbox):
#   [1, 2]   row 0 (lat > ~42.5°)    col 0 = lon 0–90°  col 1 = lon 90–180°
#   [3, 4]   row 1 (lat 0–~42.5°)
# quadbin cell = 5194902170171867135  (quadbin_tile_to_cell(1, 0, 1))
CELL_1_0_1 = 5194902170171867135  # quadbin_tile_to_cell(1, 0, 1)

if BACKEND == 'pg':
    cur.execute("""
    SELECT raster_tile_value_quadbin(
        '\\x01020304'::bytea,          -- 4 UINT8 pixels: 1,2,3,4
        2::integer,                    -- width
        2::integer,                    -- height
        5194902170171867135::bigint,   -- tile (x=1, y=0, zoom=1)
        'UINT8',
        0.0, false,
        tgeompointFromText(''SRID=4326;{Point(45 75)@2024-01-01,
                                      Point(135 75)@2024-01-02,
                                      Point(45 10)@2024-01-03,
                                      Point(-45 75)@2024-01-04}'')  -- lon -45 outside
    )::text AS result
    """)
    print('raster_tile_value_quadbin result:', cur.fetchone()[0])
    # Expected: {1@2024-01-01 ..., 2@2024-01-02 ..., 3@2024-01-03 ...}
else:
    # DuckDB path: pure-Python equivalent of raster_tile_value_quadbin.
    # The bounding box of a QUADBIN cell is fully determined by the cell id;
    # pixels are mapped to geographic coordinates via linear interpolation
    # within that bbox (matching MobilityDB's own implementation).
    import struct as _struct
    import quadbin as _qb

    def _tile_value_quadbin(pixel_bytes, width, height, cell, dtype, nodata, little_endian, instants_lnglat):
        """Python equivalent of raster_tile_value_quadbin(bytea, w, h, cell, dtype, nodata, le, tgeompoint)."""
        west, south, east, north = _qb.cell_to_bounding_box(cell)
        fmt = {'UINT8': 'B', 'UINT16': 'H', 'INT16': 'h', 'FLOAT32': 'f', 'FLOAT64': 'd'}.get(dtype, 'B')
        px  = _struct.calcsize(fmt)
        end = '<' if little_endian else '>'
        out = []
        for ts, lng, lat in instants_lnglat:
            if not (west <= lng < east and south <= lat <= north):
                continue
            col = min(int((lng - west) / (east - west) * width),  width  - 1)
            row = min(int((north - lat) / (north - south) * height), height - 1)
            val = _struct.unpack_from(end + fmt, pixel_bytes, (row * width + col) * px)[0]
            if val != nodata:
                out.append((ts, float(val)))
        return out

    instants = [
        ('2024-01-01T00:00:00+00:00',  45.0, 75.0),
        ('2024-01-02T00:00:00+00:00', 135.0, 75.0),
        ('2024-01-03T00:00:00+00:00',  45.0, 10.0),
        ('2024-01-04T00:00:00+00:00', -45.0, 75.0),  # lon -45 → outside tile
    ]
    hits = _tile_value_quadbin(bytes([1,2,3,4]), 2, 2, CELL_1_0_1, 'UINT8', 0.0, False, instants)
    parts = [f'{int(v)}@{ts[:10]}' for ts, v in hits]
    print('raster_tile_value_quadbin result:', '{' + ', '.join(parts) + '}')
    # Expected: {1@2024-01-01, 2@2024-01-02, 3@2024-01-03}


In [ ]:
if BACKEND == 'pg':
    cur.execute("DROP TABLE IF EXISTS demo_elevation")
cur.close()
conn.close()
if BACKEND == 'duckdb':
    duck.close()
print('Done.')
